# Error Analysis of Offline Inference

## Sampel Run Inspection

In [ ]:
from pathlib import Path
import os

print("Notebook is running from:")
print(Path.cwd())

In [ ]:
# Inspect single file with offline predictions for the selected custom dataset

from pathlib import Path

import numpy as np

npz_candidates = [
    Path("./old_dataset/offline_predictions.npz"),
    Path("custom_data_pipeline/old_dataset/offline_predictions.npz"),
]
npz_path = next((path for path in npz_candidates if path.exists()), None)
if npz_path is None:
    raise FileNotFoundError(
        "Could not find offline_predictions.npz. Tried: "
        + ", ".join(str(path) for path in npz_candidates)
    )

data = np.load(npz_path)

print("loaded:", npz_path)
print(data.files)

pred = data["pred_action"]
gt = data["gt_action"]

# Change this one value to compare trajectory plots across demos.
DEMO_ID_TO_PLOT = 0
print("demo selected for trajectory plots:", DEMO_ID_TO_PLOT)

In [ ]:
sample_idx = 0

print("Metadata, sample 0:")
print("dataset_index:", data["dataset_index"][sample_idx])
print("episode_id:", data["episode_id"][sample_idx])
print("episode_timestep:", data["episode_timestep"][sample_idx])
print("before_first_grasp:", data["before_first_grasp"][sample_idx])

print("\nPrediction, sample 0:")
print(pred[sample_idx])

print("\nGround truth, sample 0:")
print(gt[sample_idx])


## Error Calculation

In [ ]:
# Overall error metrics
# err has the same shape as pred and gt: (samples, future_timesteps, action_dims)

err = pred - gt
abs_err = np.abs(err)

overall_mae = abs_err.mean()
overall_rmse = np.sqrt((err ** 2).mean())

print("pred shape:", pred.shape)
print("gt shape:", gt.shape)
print("Overall MAE:", overall_mae)
print("Overall RMSE:", overall_rmse)

In [ ]:
# MAE by action group
# UMI action layout:
# dims 0:3 = position
# dims 3:9 = raw 6D rotation representation
# dim 9 = gripper width

pos_mae = abs_err[..., 0:3].mean()
rot6d_mae = abs_err[..., 3:9].mean()
gripper_mae = abs_err[..., 9].mean()

print("Position MAE:", pos_mae)
print("Raw 6D rotation MAE:", rot6d_mae)
print("Gripper MAE:", gripper_mae)

In [ ]:
# Error as the predicted future horizon increases
# x-axis is future timestep: 0 is the first predicted action, 15 is the last.

import matplotlib.pyplot as plt

horizon_steps = np.arange(pred.shape[1])

overall_mae_by_horizon = abs_err.mean(axis=(0, 2))
pos_mae_by_horizon = abs_err[..., 0:3].mean(axis=(0, 2))
rot6d_mae_by_horizon = abs_err[..., 3:9].mean(axis=(0, 2))
gripper_mae_by_horizon = abs_err[..., 9].mean(axis=0)

plt.figure(figsize=(10, 5))
plt.plot(horizon_steps, overall_mae_by_horizon, marker="o", label="overall MAE")
plt.plot(horizon_steps, pos_mae_by_horizon, marker="o", label="position MAE")
plt.plot(horizon_steps, rot6d_mae_by_horizon, marker="o", label="raw 6D rotation MAE")
plt.plot(horizon_steps, gripper_mae_by_horizon, marker="o", label="gripper MAE")
plt.xlabel("future horizon timestep")
plt.ylabel("MAE")
plt.title("Error vs predicted future horizon")
plt.xticks(horizon_steps)
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

In [ ]:
# Immediate prediction error throughout each demo
# x-axis: normalized timestep within demo, where 0 = start and 1 = end
# y-axis: MAE for horizon step 0 only, averaged across the 10 action dimensions

episode_id = data["episode_id"]
episode_timestep = data["episode_timestep"]

immediate_mae = abs_err[:, 0, :].mean(axis=1)

plt.figure(figsize=(12, 5))

for demo_id in np.unique(episode_id):
    mask = episode_id == demo_id
    order = np.argsort(episode_timestep[mask])

    x = episode_timestep[mask][order]
    y = immediate_mae[mask][order]

    x_norm = x / max(x.max(), 1)
    plt.plot(x_norm, y, alpha=0.2)

plt.xlabel("normalized timestep within demo")
plt.ylabel("immediate prediction MAE")
plt.title("Immediate prediction error throughout each demo")
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Average immediate prediction error across demos on normalized demo time
# x is normalized from 0 to 1 for every demo.
# y is still the actual error value; we average y across demos at each normalized x location.

norm_grid = np.linspace(0, 1, 101)

immediate_pos_mae = abs_err[:, 0, 0:3].mean(axis=1)
immediate_rot6d_mae = abs_err[:, 0, 3:9].mean(axis=1)
immediate_gripper_mae = abs_err[:, 0, 9]

pos_curves = []
rot6d_curves = []
gripper_curves = []

for demo_id in np.unique(episode_id):
    mask = episode_id == demo_id
    if mask.sum() < 2:
        continue

    order = np.argsort(episode_timestep[mask])
    x = episode_timestep[mask][order]
    x_norm = x / max(x.max(), 1)

    pos_curves.append(np.interp(norm_grid, x_norm, immediate_pos_mae[mask][order]))
    rot6d_curves.append(np.interp(norm_grid, x_norm, immediate_rot6d_mae[mask][order]))
    gripper_curves.append(np.interp(norm_grid, x_norm, immediate_gripper_mae[mask][order]))

mean_pos_curve = np.mean(pos_curves, axis=0)
mean_rot6d_curve = np.mean(rot6d_curves, axis=0)
mean_gripper_curve = np.mean(gripper_curves, axis=0)

plt.figure(figsize=(12, 5))
plt.plot(norm_grid, mean_pos_curve, label="position MAE")
plt.plot(norm_grid, mean_rot6d_curve, label="raw 6D rotation MAE")
plt.plot(norm_grid, mean_gripper_curve, label="gripper MAE")
plt.xlabel("normalized timestep within demo")
plt.ylabel("average immediate prediction error")
plt.title("Average immediate prediction error over normalized demo time")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

In [ ]:
# Average prediction error over normalized demo time, split by prediction horizon
# For sample i and horizon h, pred[i, h] is compared with gt[i, h].
# We average the error inside each action group, align each demo to x=0..1,
# then average those normalized demo curves separately for every horizon step.
# Horizon 0 is the immediate prediction. Future horizon steps fade lighter as they get farther out.

import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

abs_err = np.abs(pred - gt)
episode_id = data["episode_id"]
episode_timestep = data["episode_timestep"]
demo_ids = np.unique(episode_id)
norm_grid = np.linspace(0, 1, 101)
num_horizon_steps = pred.shape[1]

def average_over_normalized_demo_time(error_by_sample_horizon):
    mean_curves = []

    for horizon_idx in range(num_horizon_steps):
        demo_curves = []

        for demo_id in demo_ids:
            mask = episode_id == demo_id
            if mask.sum() < 2:
                continue

            order = np.argsort(episode_timestep[mask])
            x = episode_timestep[mask][order]
            y = error_by_sample_horizon[mask, horizon_idx][order]
            x_norm = x / max(x.max(), 1)

            demo_curves.append(np.interp(norm_grid, x_norm, y))

        mean_curves.append(np.mean(demo_curves, axis=0))

    return np.array(mean_curves)

overall_error_by_horizon = abs_err.mean(axis=2)
pos_error_by_horizon = abs_err[..., 0:3].mean(axis=2)
rot6d_error_by_horizon = abs_err[..., 3:9].mean(axis=2)
gripper_error_by_horizon = abs_err[..., 9]

plots = [
    ("overall MAE", average_over_normalized_demo_time(overall_error_by_horizon)),
    ("position MAE", average_over_normalized_demo_time(pos_error_by_horizon)),
    ("raw 6D rotation MAE", average_over_normalized_demo_time(rot6d_error_by_horizon)),
    ("gripper MAE", average_over_normalized_demo_time(gripper_error_by_horizon)),
]

future_cmap = plt.cm.Blues_r
future_denominator = max(num_horizon_steps - 2, 1)

fig, axes = plt.subplots(2, 2, figsize=(14, 8), sharex=True, constrained_layout=True)

for ax, (title, curves) in zip(axes.ravel(), plots):
    for horizon_idx in range(num_horizon_steps - 1, 0, -1):
        fade = (horizon_idx - 1) / future_denominator
        color = future_cmap(0.15 + 0.70 * fade)
        alpha = 0.85 - 0.45 * fade
        ax.plot(norm_grid, curves[horizon_idx], color=color, alpha=alpha, linewidth=1.2)

    ax.plot(norm_grid, curves[0], color="black", linewidth=2.6)
    ax.set_title(title)
    ax.set_ylabel("MAE")
    ax.grid(True, alpha=0.3)

for ax in axes[-1, :]:
    ax.set_xlabel("normalized timestep within demo")

legend_handles = [
    Line2D([0], [0], color="black", linewidth=2.6, label="horizon 0 / immediate"),
    Line2D([0], [0], color=future_cmap(0.35), linewidth=1.5, label="future horizons / lighter = farther"),
]
axes[0, 0].legend(handles=legend_handles, loc="best")

if num_horizon_steps > 1:
    norm = plt.Normalize(vmin=1, vmax=num_horizon_steps - 1)
    scalar_map = plt.cm.ScalarMappable(cmap=future_cmap, norm=norm)
    scalar_map.set_array([])
    colorbar = fig.colorbar(scalar_map, ax=axes.ravel().tolist(), shrink=0.9, pad=0.02)
    colorbar.set_label("future prediction horizon step")

fig.suptitle("Average prediction error across demos by prediction horizon")
plt.show()

## Mapping of Prediction and Ground Truth

In [ ]:
# Pick 5 demos whose full-horizon MAE is closest to the average demo MAE.
# This gives examples that are not cherry-picked as especially good or especially bad.

per_sample_full_horizon_mae = abs_err.mean(axis=(1, 2))
demo_error_rows = []

for demo_id in np.unique(episode_id):
    mask = episode_id == demo_id
    if mask.sum() == 0:
        continue

    demo_mae = per_sample_full_horizon_mae[mask].mean()
    demo_error_rows.append((int(demo_id), float(demo_mae), int(mask.sum())))

demo_error_rows = np.array(
    demo_error_rows,
    dtype=[("episode_id", int), ("mae", float), ("num_samples", int)],
)

mean_demo_mae = demo_error_rows["mae"].mean()
closest_to_average = np.argsort(np.abs(demo_error_rows["mae"] - mean_demo_mae))
average_demo_ids = demo_error_rows["episode_id"][closest_to_average[:5]]

print("mean per-demo full-horizon MAE:", mean_demo_mae)
print("selected average-ish demos:")

for demo_id in average_demo_ids:
    row = demo_error_rows[demo_error_rows["episode_id"] == demo_id][0]
    print(f"demo {row['episode_id']}: MAE={row['mae']:.6f}, samples={row['num_samples']}")

In [ ]:
# Map ground-truth and predicted future branches.
# Each branch is one sample's full 16-step future horizon.
# Ground-truth future branches are black.
# Predicted future branches are blue and start from the sample's current ground-truth value.
# Horizon 0 prediction is shown as an orange marker at the branch root.
# We do not connect horizon-1 values across the whole demo, because each forecast starts from a different sample.

from matplotlib.lines import Line2D

# From the training config: action.down_sample_steps = task.obs_down_sample_steps = 3.
# So pred[i, h] corresponds to episode_timestep[i] + h * 3 in replay-buffer time.
action_down_sample_steps = 3

action_dims = list(range(9))
action_dim_names = [
    "pos x", "pos y", "pos z",
    "rot6d 0", "rot6d 1", "rot6d 2",
    "rot6d 3", "rot6d 4", "rot6d 5",
]

def plot_demo_value_branches(demo_id, num_branch_anchors=35):
    mask = episode_id == demo_id
    order = np.argsort(episode_timestep[mask])
    sample_indices = np.where(mask)[0][order]

    if len(sample_indices) == 0:
        print(f"demo {demo_id}: no samples found")
        return

    t = episode_timestep[sample_indices]
    horizon_offsets = np.arange(num_horizon_steps) * action_down_sample_steps
    x_max = max(t[-1] + horizon_offsets[-1], 1)
    anchor_positions = np.unique(
        np.linspace(0, len(sample_indices) - 1, min(num_branch_anchors, len(sample_indices)), dtype=int)
    )

    fig, axes = plt.subplots(3, 3, figsize=(16, 9), sharex=True, constrained_layout=True)

    for ax, dim, dim_name in zip(axes.ravel(), action_dims, action_dim_names):
        for anchor_pos in anchor_positions:
            sample_idx = sample_indices[anchor_pos]
            branch_x = (episode_timestep[sample_idx] + horizon_offsets) / x_max
            gt_branch_y = gt[sample_idx, :, dim]
            pred_branch_y = np.concatenate([[gt[sample_idx, 0, dim]], pred[sample_idx, 1:, dim]])

            ax.plot(branch_x, gt_branch_y, color="black", linewidth=1.0, alpha=0.35)
            ax.scatter(branch_x, gt_branch_y, color="black", s=5, alpha=0.22)

            ax.plot(branch_x, pred_branch_y, color="tab:blue", linewidth=0.9, alpha=0.28)
            ax.scatter(branch_x[1:], pred_branch_y[1:], color="tab:blue", s=5, alpha=0.22)
            ax.scatter(branch_x[0], pred[sample_idx, 0, dim], color="tab:orange", s=12, alpha=0.45)

        ax.set_title(dim_name)
        ax.grid(True, alpha=0.25)
        ax.set_ylabel("action value")

    for ax in axes[-1, :]:
        ax.set_xlabel("normalized timestep within demo")

    legend_handles = [
        Line2D([0], [0], color="black", linewidth=1.4, label="ground-truth future branch, h=0..15"),
        Line2D([0], [0], color="tab:blue", linewidth=1.4, alpha=0.7, label="predicted future branch, h=1..15"),
        Line2D([0], [0], marker="o", color="tab:orange", linestyle="None", markersize=5, label="immediate prediction, h=0"),
    ]

    fig.legend(handles=legend_handles, loc="upper center", ncol=3)
    fig.suptitle(f"Demo {demo_id}: ground truth and prediction branches", y=1.03)
    plt.show()

demo_ids_to_plot = [int(DEMO_ID_TO_PLOT)] if "DEMO_ID_TO_PLOT" in globals() else [int(x) for x in average_demo_ids]
for demo_id in demo_ids_to_plot:
    plot_demo_value_branches(demo_id)

In [ ]:
# Ground-truth and prediction horizon branch maps for one average-ish demo.
# This uses side-by-side panels instead of overlaying GT and prediction.
# Left column: ground-truth branches. Right column: prediction branches.
# Each row uses the same y-axis limits, so the shapes and scale are directly comparable.
# Each branch is one sample timestamp: horizon 0 -> horizon 1 -> ... -> horizon 15.

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection
from matplotlib.lines import Line2D

if "action_down_sample_steps" not in globals():
    action_down_sample_steps = 3

if "num_horizon_steps" not in globals():
    num_horizon_steps = gt.shape[1]

if "action_dims" not in globals():
    action_dims = list(range(9))
    action_dim_names = [
        "pos x", "pos y", "pos z",
        "rot6d 0", "rot6d 1", "rot6d 2",
        "rot6d 3", "rot6d 4", "rot6d 5",
    ]

if "average_demo_ids" not in globals():
    abs_err = np.abs(pred - gt)
    per_sample_full_horizon_mae = abs_err.mean(axis=(1, 2))
    demo_error_rows = []

    for this_demo_id in np.unique(episode_id):
        mask = episode_id == this_demo_id
        demo_error_rows.append((int(this_demo_id), float(per_sample_full_horizon_mae[mask].mean())))

    demo_error_rows = np.array(demo_error_rows, dtype=[("episode_id", int), ("mae", float)])
    mean_demo_mae = demo_error_rows["mae"].mean()
    closest_to_average = np.argsort(np.abs(demo_error_rows["mae"] - mean_demo_mae))
    average_demo_ids = demo_error_rows["episode_id"][closest_to_average[:5]]

demo_id = int(DEMO_ID_TO_PLOT) if "DEMO_ID_TO_PLOT" in globals() else int(average_demo_ids[0])
mask = episode_id == demo_id
order = np.argsort(episode_timestep[mask])
sample_indices = np.where(mask)[0][order]
if len(sample_indices) == 0:
    raise ValueError(f"DEMO_ID_TO_PLOT={demo_id} was not found in episode_id")

horizon_offsets = np.arange(num_horizon_steps) * action_down_sample_steps
x_max = max(episode_timestep[sample_indices[-1]] + horizon_offsets[-1], 1)

num_branch_roots = 25
branch_positions = np.unique(
    np.linspace(0, len(sample_indices) - 1, min(num_branch_roots, len(sample_indices)), dtype=int)
)
branch_sample_indices = sample_indices[branch_positions]

norm = plt.Normalize(vmin=0, vmax=num_horizon_steps - 1)
cmap = plt.cm.viridis

def draw_branch_map(ax, values, dim, title, y_limits):
    branch_lines = []
    all_x = []
    all_y = []
    all_horizon_ids = []
    root_x = []
    root_y = []

    for sample_idx in branch_sample_indices:
        branch_x = (episode_timestep[sample_idx] + horizon_offsets) / x_max
        branch_y = values[sample_idx, :, dim]

        branch_lines.append(np.column_stack([branch_x, branch_y]))
        all_x.append(branch_x)
        all_y.append(branch_y)
        all_horizon_ids.append(np.arange(num_horizon_steps))
        root_x.append(branch_x[0])
        root_y.append(branch_y[0])

    ax.add_collection(LineCollection(branch_lines, colors="0.25", linewidths=0.9, alpha=0.35))
    scatter = ax.scatter(
        np.concatenate(all_x),
        np.concatenate(all_y),
        c=np.concatenate(all_horizon_ids),
        cmap=cmap,
        norm=norm,
        s=16,
        alpha=0.8,
        linewidths=0,
    )
    ax.scatter(root_x, root_y, color="tab:orange", s=32, alpha=0.95, linewidths=0)

    ax.set_xlim(0, 1)
    ax.set_ylim(*y_limits)
    ax.set_title(title)
    ax.grid(True, alpha=0.25)
    return scatter

fig, axes = plt.subplots(
    len(action_dims),
    2,
    figsize=(13, 24),
    sharex=True,
    sharey="row",
    constrained_layout=True,
)

scatter = None
for row_idx, (dim, dim_name) in enumerate(zip(action_dims, action_dim_names)):
    gt_values = gt[branch_sample_indices, :, dim].reshape(-1)
    pred_values = pred[branch_sample_indices, :, dim].reshape(-1)
    combined_values = np.concatenate([gt_values, pred_values])
    value_min = combined_values.min()
    value_max = combined_values.max()
    padding = max((value_max - value_min) * 0.08, 1e-4)
    y_limits = (value_min - padding, value_max + padding)

    scatter = draw_branch_map(axes[row_idx, 0], gt, dim, f"{dim_name} ground truth", y_limits)
    draw_branch_map(axes[row_idx, 1], pred, dim, f"{dim_name} prediction", y_limits)
    axes[row_idx, 0].set_ylabel("relative action value")

for ax in axes[-1, :]:
    ax.set_xlabel("normalized demo time where target lands")

colorbar = fig.colorbar(scatter, ax=axes.ravel().tolist(), shrink=0.9, pad=0.02)
colorbar.set_label("horizon step within each branch")

legend_handles = [
    Line2D([0], [0], color="0.25", linewidth=1.2, label="branch path"),
    Line2D([0], [0], marker="o", color="tab:orange", linestyle="None", markersize=5, label="branch root, h=0"),
]
fig.legend(handles=legend_handles, loc="upper center", ncol=2)

fig.suptitle(
    f"Demo {demo_id}: side-by-side GT vs prediction branches ({len(branch_sample_indices)} roots)",
    y=1.01,
)
plt.show()

print(f"demo {demo_id}: plotted {len(branch_sample_indices)} branch roots, side-by-side GT and prediction")


In [ ]:
# Best-case and worst-case demos by per-demo full-horizon MAE.
# Uses the same side-by-side branch map as above: left = GT, right = prediction.

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection
from matplotlib.lines import Line2D

action_down_sample_steps = 3
num_horizon_steps = gt.shape[1]
action_dims = list(range(9))
action_dim_names = [
    "pos x", "pos y", "pos z",
    "rot6d 0", "rot6d 1", "rot6d 2",
    "rot6d 3", "rot6d 4", "rot6d 5",
]

abs_err = np.abs(pred - gt)
per_sample_full_horizon_mae = abs_err.mean(axis=(1, 2))
demo_error_rows = []

for this_demo_id in np.unique(episode_id):
    mask = episode_id == this_demo_id
    demo_error_rows.append((int(this_demo_id), float(per_sample_full_horizon_mae[mask].mean()), int(mask.sum())))

demo_error_rows = np.array(
    demo_error_rows,
    dtype=[("episode_id", int), ("mae", float), ("num_samples", int)],
)

best_demo_row = demo_error_rows[np.argmin(demo_error_rows["mae"])]
worst_demo_row = demo_error_rows[np.argmax(demo_error_rows["mae"])]

print("best demo:", int(best_demo_row["episode_id"]), "MAE:", best_demo_row["mae"], "samples:", int(best_demo_row["num_samples"]))
print("worst demo:", int(worst_demo_row["episode_id"]), "MAE:", worst_demo_row["mae"], "samples:", int(worst_demo_row["num_samples"]))

def plot_side_by_side_branch_map(demo_id, title_prefix, num_branch_roots=25):
    mask = episode_id == demo_id
    order = np.argsort(episode_timestep[mask])
    sample_indices = np.where(mask)[0][order]

    horizon_offsets = np.arange(num_horizon_steps) * action_down_sample_steps
    x_max = max(episode_timestep[sample_indices[-1]] + horizon_offsets[-1], 1)

    branch_positions = np.unique(
        np.linspace(0, len(sample_indices) - 1, min(num_branch_roots, len(sample_indices)), dtype=int)
    )
    branch_sample_indices = sample_indices[branch_positions]

    norm = plt.Normalize(vmin=0, vmax=num_horizon_steps - 1)
    cmap = plt.cm.viridis

    def draw_branch_map(ax, values, dim, title, y_limits):
        branch_lines = []
        all_x = []
        all_y = []
        all_horizon_ids = []
        root_x = []
        root_y = []

        for sample_idx in branch_sample_indices:
            branch_x = (episode_timestep[sample_idx] + horizon_offsets) / x_max
            branch_y = values[sample_idx, :, dim]

            branch_lines.append(np.column_stack([branch_x, branch_y]))
            all_x.append(branch_x)
            all_y.append(branch_y)
            all_horizon_ids.append(np.arange(num_horizon_steps))
            root_x.append(branch_x[0])
            root_y.append(branch_y[0])

        ax.add_collection(LineCollection(branch_lines, colors="0.25", linewidths=0.9, alpha=0.35))
        scatter = ax.scatter(
            np.concatenate(all_x),
            np.concatenate(all_y),
            c=np.concatenate(all_horizon_ids),
            cmap=cmap,
            norm=norm,
            s=16,
            alpha=0.8,
            linewidths=0,
        )
        ax.scatter(root_x, root_y, color="tab:orange", s=32, alpha=0.95, linewidths=0)

        ax.set_xlim(0, 1)
        ax.set_ylim(*y_limits)
        ax.set_title(title)
        ax.grid(True, alpha=0.25)
        return scatter

    fig, axes = plt.subplots(
        len(action_dims),
        2,
        figsize=(13, 24),
        sharex=True,
        sharey="row",
        constrained_layout=True,
    )

    scatter = None
    for row_idx, (dim, dim_name) in enumerate(zip(action_dims, action_dim_names)):
        gt_values = gt[branch_sample_indices, :, dim].reshape(-1)
        pred_values = pred[branch_sample_indices, :, dim].reshape(-1)
        combined_values = np.concatenate([gt_values, pred_values])
        value_min = combined_values.min()
        value_max = combined_values.max()
        padding = max((value_max - value_min) * 0.08, 1e-4)
        y_limits = (value_min - padding, value_max + padding)

        scatter = draw_branch_map(axes[row_idx, 0], gt, dim, f"{dim_name} ground truth", y_limits)
        draw_branch_map(axes[row_idx, 1], pred, dim, f"{dim_name} prediction", y_limits)
        axes[row_idx, 0].set_ylabel("relative action value")

    for ax in axes[-1, :]:
        ax.set_xlabel("normalized demo time where target lands")

    colorbar = fig.colorbar(scatter, ax=axes.ravel().tolist(), shrink=0.9, pad=0.02)
    colorbar.set_label("horizon step within each branch")

    legend_handles = [
        Line2D([0], [0], color="0.25", linewidth=1.2, label="branch path"),
        Line2D([0], [0], marker="o", color="tab:orange", linestyle="None", markersize=5, label="branch root, h=0"),
    ]
    fig.legend(handles=legend_handles, loc="upper center", ncol=2)

    fig.suptitle(
        f"{title_prefix} demo {demo_id}: side-by-side GT vs prediction branches ({len(branch_sample_indices)} roots)",
        y=1.01,
    )
    plt.show()

plot_side_by_side_branch_map(int(best_demo_row["episode_id"]), "Best-case")
plot_side_by_side_branch_map(int(worst_demo_row["episode_id"]), "Worst-case")

## Legacy Absolute 3D Trajectory Visualization

This section is self-contained. Start running from the code cell directly below, change `DEMO_ID_TO_PLOT` there, then run the following Vedo plot cell. It reloads the offline inference output into separate `abs3d_*` variables and converts relative action labels/predictions back into absolute end-effector poses for 3D plotting.


In [1]:
# Self-contained setup for the legacy absolute 3D trajectory section.
# Change DEMO_ID_TO_PLOT here, then run this cell and the Vedo plot cell below.
# This cell expects an offline_inference output that includes:
# base_eef_pos and base_eef_rot_axis_angle.

import json
import sys
from pathlib import Path

import numpy as np

# ===== Section config =====
DEMO_ID_TO_PLOT = 0
ABS3D_NPZ_CANDIDATES = [
    Path("./old_dataset/offline_predictions.npz"),
    Path("custom_data_pipeline/old_dataset/offline_predictions.npz"),
]

abs3d_npz_candidates = ABS3D_NPZ_CANDIDATES
abs3d_npz_path = next((path for path in abs3d_npz_candidates if path.exists()), None)

if abs3d_npz_path is None:
    raise FileNotFoundError(
        "Could not find offline_predictions.npz. Tried: "
        + ", ".join(str(path) for path in abs3d_npz_candidates)
    )

abs3d_results_dir = abs3d_npz_path.parent
abs3d_metadata_path = abs3d_results_dir / "metadata.json"

abs3d_data = np.load(abs3d_npz_path)
abs3d_required_keys = [
    "pred_action",
    "gt_action",
    "episode_id",
    "episode_timestep",
    "base_eef_pos",
    "base_eef_rot_axis_angle",
]
abs3d_missing_keys = [key for key in abs3d_required_keys if key not in abs3d_data.files]

if abs3d_missing_keys:
    raise KeyError(
        "This result file is missing keys needed for absolute 3D plotting: "
        + str(abs3d_missing_keys)
        + "\nRerun offline_inference.py after the base_eef_pos/base_eef_rot_axis_angle save change."
    )

if abs3d_metadata_path.exists():
    with open(abs3d_metadata_path, "r") as f:
        abs3d_metadata = json.load(f)
else:
    abs3d_metadata = {}

abs3d_action_down_sample_steps = int(abs3d_metadata.get("action_down_sample_steps", 3))

# The notebook can be run from either the repo root or test_inference/.
# Add the repo root to sys.path so umi.common.pose_util imports correctly.
abs3d_repo_root = None
for abs3d_candidate_root in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]:
    if (abs3d_candidate_root / "umi" / "common" / "pose_util.py").exists():
        abs3d_repo_root = abs3d_candidate_root
        break

if abs3d_repo_root is None:
    raise FileNotFoundError("Could not locate the repo root containing umi/common/pose_util.py")

if str(abs3d_repo_root) not in sys.path:
    sys.path.insert(0, str(abs3d_repo_root))

from umi.common.pose_util import pose_to_mat, pose10d_to_mat

abs3d_pred_action = abs3d_data["pred_action"]
abs3d_gt_action = abs3d_data["gt_action"]
abs3d_episode_id = abs3d_data["episode_id"]
abs3d_episode_timestep = abs3d_data["episode_timestep"]
abs3d_base_eef_pos = abs3d_data["base_eef_pos"]
abs3d_base_eef_rot_axis_angle = abs3d_data["base_eef_rot_axis_angle"]

# Action layout is [x, y, z, rot6d(6 values), gripper].
# pose10d_to_mat uses only the first 9 values: position + 6D rotation.
# The labels/predictions are relative to the current end-effector pose, so:
# absolute_future_pose = absolute_current_pose @ relative_future_pose
abs3d_base_pose = np.concatenate(
    [abs3d_base_eef_pos, abs3d_base_eef_rot_axis_angle],
    axis=-1,
)
abs3d_base_pose_mat = pose_to_mat(abs3d_base_pose)

# The repo helper pose10d_to_mat expects a flat batch of poses, so flatten
# (samples, horizon, 9) -> (samples * horizon, 9), convert, then reshape.
def abs3d_action_pose_to_mat(abs3d_action_pose):
    abs3d_flat_pose = abs3d_action_pose.reshape(-1, abs3d_action_pose.shape[-1])
    abs3d_flat_pose_mat = pose10d_to_mat(abs3d_flat_pose)
    return abs3d_flat_pose_mat.reshape(abs3d_action_pose.shape[:-1] + (4, 4))

abs3d_gt_rel_pose_mat = abs3d_action_pose_to_mat(abs3d_gt_action[..., :9])
abs3d_pred_rel_pose_mat = abs3d_action_pose_to_mat(abs3d_pred_action[..., :9])

# ABSOLUTE FUTURE EEF POSE RECONSTRUCTION:
# This mirrors convert_pose_mat_rep(..., pose_rep="relative", backward=True)
# in diffusion_policy/common/pose_repr_util.py, where relative actions are
# converted back with:
#     absolute_pose_mat = base_pose_mat @ relative_pose_mat
# Position comes from the matrix translation column; orientation comes from
# the matrix rotation block.
abs3d_gt_pose_mat = abs3d_base_pose_mat[:, None, :, :] @ abs3d_gt_rel_pose_mat
abs3d_pred_pose_mat = abs3d_base_pose_mat[:, None, :, :] @ abs3d_pred_rel_pose_mat

abs3d_gt_xyz = abs3d_gt_pose_mat[..., :3, 3]
abs3d_pred_xyz = abs3d_pred_pose_mat[..., :3, 3]

abs3d_demo_ids = np.unique(abs3d_episode_id)
if len(abs3d_demo_ids) == 0:
    raise ValueError("No demos were found in the offline inference output.")

abs3d_demo_id = int(DEMO_ID_TO_PLOT) if "DEMO_ID_TO_PLOT" in globals() else int(abs3d_demo_ids[0])
if abs3d_demo_id not in set(int(x) for x in abs3d_demo_ids):
    raise ValueError(f"DEMO_ID_TO_PLOT={abs3d_demo_id} was not found. Available demos: {abs3d_demo_ids.tolist()}")
if len(abs3d_demo_ids) == 1:
    print("Only one episode is present; using it, but it may be partial.")

abs3d_demo_mask = abs3d_episode_id == abs3d_demo_id
abs3d_demo_order = np.argsort(abs3d_episode_timestep[abs3d_demo_mask])
abs3d_demo_sample_indices = np.where(abs3d_demo_mask)[0][abs3d_demo_order]

print("loaded:", abs3d_npz_path)
print("available keys:", abs3d_data.files)
print("selected demo id:", abs3d_demo_id)
print("samples in selected demo:", len(abs3d_demo_sample_indices))
print("prediction horizon:", abs3d_pred_action.shape[1])
print("action downsample steps:", abs3d_action_down_sample_steps)
print("absolute GT xyz shape:", abs3d_gt_xyz.shape)
print("absolute prediction xyz shape:", abs3d_pred_xyz.shape)


loaded: old_dataset/offline_predictions.npz
available keys: ['pred_action', 'gt_action', 'dataset_index', 'episode_id', 'episode_timestep', 'before_first_grasp', 'base_eef_pos', 'base_eef_rot_axis_angle']
selected demo id: 0
samples in selected demo: 564
prediction horizon: 16
action downsample steps: 3
absolute GT xyz shape: (6012, 16, 3)
absolute prediction xyz shape: (6012, 16, 3)


In [ ]:
# 3D Vedo plot for DEMO_ID_TO_PLOT.
# Black line: actual absolute end-effector path from the saved current/base poses.
# Black dots: root timestamps where we start prediction horizons.
# Orange gradient branches: predicted future horizons from each black dot.
# Light gray branches: ground-truth future horizons from each black dot.
# Arrows: red = local x axis, green = local y axis, blue = local z axis.

try:
    import vedo
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        "vedo is not installed in this environment. "
        "Activate blackwell-train, then run: pip install vedo"
    ) from exc

import matplotlib.pyplot as plt
from matplotlib.colors import to_hex

abs3d_num_branch_roots = 25
abs3d_root_positions = np.unique(
    np.linspace(
        0,
        len(abs3d_demo_sample_indices) - 1,
        min(abs3d_num_branch_roots, len(abs3d_demo_sample_indices)),
        dtype=int,
    )
)
abs3d_root_indices = abs3d_demo_sample_indices[abs3d_root_positions]


def abs3d_make_line(points, color, linewidth=2, alpha=1.0):
    try:
        return vedo.Line(points, c=color, lw=linewidth, alpha=alpha)
    except TypeError:
        line = vedo.Line(points, c=color, lw=linewidth)
        if hasattr(line, "alpha"):
            line.alpha(alpha)
        return line


def abs3d_make_points(points, color, radius=6, alpha=1.0):
    try:
        return vedo.Points(points, c=color, r=radius, alpha=alpha)
    except TypeError:
        pts = vedo.Points(points, c=color, r=radius)
        if hasattr(pts, "alpha"):
            pts.alpha(alpha)
        return pts


def abs3d_make_arrows(starts, ends, color, alpha=1.0):
    try:
        return vedo.Arrows(starts, ends, c=color, alpha=alpha)
    except TypeError:
        arrows = vedo.Arrows(starts, ends, c=color)
        if hasattr(arrows, "alpha"):
            arrows.alpha(alpha)
        return arrows


def abs3d_make_gradient_line(points, cmap_name="Oranges", linewidth=3, alpha=0.9):
    # Draw one small line segment per horizon step.
    # Dark orange is near the root; lighter orange is farther into the future.
    cmap = plt.get_cmap(cmap_name)
    num_segments = max(len(points) - 1, 1)
    color_values = np.linspace(0.95, 0.35, num_segments)

    segment_actors = []
    for segment_idx, color_value in enumerate(color_values):
        segment_points = points[segment_idx:segment_idx + 2]
        segment_color = to_hex(cmap(color_value))
        segment_actors.append(
            abs3d_make_line(segment_points, segment_color, linewidth=linewidth, alpha=alpha)
        )

    return segment_actors


def abs3d_axis_arrows(pose_mats, every=5, scale=0.025, alpha=0.75):
    starts = pose_mats[::every, :3, 3]
    rotations = pose_mats[::every, :3, :3]

    if len(starts) == 0:
        return []

    x_ends = starts + rotations[:, :, 0] * scale
    y_ends = starts + rotations[:, :, 1] * scale
    z_ends = starts + rotations[:, :, 2] * scale

    return [
        abs3d_make_arrows(starts, x_ends, "red", alpha=alpha),
        abs3d_make_arrows(starts, y_ends, "green", alpha=alpha),
        abs3d_make_arrows(starts, z_ends, "blue", alpha=alpha),
    ]


abs3d_actors = []

# The actual path is the current absolute end-effector position at each sample.
abs3d_demo_actual_xyz = abs3d_base_eef_pos[abs3d_demo_sample_indices]
abs3d_root_xyz = abs3d_base_eef_pos[abs3d_root_indices]

abs3d_actors.append(abs3d_make_line(abs3d_demo_actual_xyz, "black", linewidth=5, alpha=0.95))
abs3d_actors.append(abs3d_make_points(abs3d_root_xyz, "black", radius=9, alpha=1.0))

# Each branch starts at the black root point.
for abs3d_root_idx in abs3d_root_indices:
    abs3d_root_point = abs3d_base_eef_pos[abs3d_root_idx][None, :]
    abs3d_gt_branch_xyz = np.vstack([abs3d_root_point, abs3d_gt_xyz[abs3d_root_idx]])
    abs3d_pred_branch_xyz = np.vstack([abs3d_root_point, abs3d_pred_xyz[abs3d_root_idx]])

    abs3d_actors.append(abs3d_make_line(abs3d_gt_branch_xyz, "lightgray", linewidth=2, alpha=0.45))
    abs3d_actors.extend(abs3d_make_gradient_line(abs3d_pred_branch_xyz, linewidth=3, alpha=0.9))

# Show orientation for a subset of the branch roots to keep the plot readable.
abs3d_orientation_root_stride = max(len(abs3d_root_indices) // 6, 1)
for abs3d_root_idx in abs3d_root_indices[::abs3d_orientation_root_stride]:
    abs3d_actors.extend(abs3d_axis_arrows(abs3d_gt_pose_mat[abs3d_root_idx], every=5, scale=0.025, alpha=0.75))
    abs3d_actors.extend(abs3d_axis_arrows(abs3d_pred_pose_mat[abs3d_root_idx], every=5, scale=0.025, alpha=0.35))

print("Vedo plot contents:")
print("- black thick line: actual absolute end-effector path")
print("- black dots: root timestamps")
print("- orange gradient branches: predicted future horizons from each black dot")
print("  darker orange = earlier horizon; lighter orange = farther future")
print("- light gray branches: ground-truth future horizons from each black dot")
print("- red/green/blue arrows: local x/y/z orientation axes")

vedo.settings.default_backend = "vtk"

vedo.show(
    abs3d_actors,
    axes=1,
    viewup="z",
    bg="white",
    title=f"Absolute 3D Trajectory: Demo {abs3d_demo_id}",
)


Vedo plot contents:
- black thick line: actual absolute end-effector path
- black dots: root timestamps
- orange gradient branches: predicted future horizons from each black dot
  darker orange = earlier horizon; lighter orange = farther future
- light gray branches: ground-truth future horizons from each black dot
- red/green/blue arrows: local x/y/z orientation axes


## Actual End-Effector 3D Pose Only

This section is self-contained. Start running from the code cell directly below and change `DEMO_ID_TO_PLOT` there. It uses only the saved actual end-effector position and rotation, and does not use predictions or ground-truth action horizons.


In [ ]:
# Self-contained setup and plot for the actual EEF 3D pose section.
# Change DEMO_ID_TO_PLOT here, then run this cell.
# Actual 3D end-effector path with orientation arrows.
# This uses only the saved current/base end-effector pose:
#   base_eef_pos
#   base_eef_rot_axis_angle
# It does not use pred_action or gt_action.

from pathlib import Path

import numpy as np
from scipy.spatial.transform import Rotation as R

try:
    import vedo
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        "vedo is not installed in this environment. "
        "Activate blackwell-train, then run: pip install vedo"
    ) from exc

# ===== Section config =====
DEMO_ID_TO_PLOT = 0
ACTUAL_NPZ_CANDIDATES = [
    Path("./old_dataset/offline_predictions.npz"),
    Path("custom_data_pipeline/old_dataset/offline_predictions.npz"),
]

npz_candidates = ACTUAL_NPZ_CANDIDATES
npz_path = next((path for path in npz_candidates if path.exists()), None)

if npz_path is None:
    raise FileNotFoundError(
        "Could not find offline_predictions.npz. Tried: "
        + ", ".join(str(path) for path in npz_candidates)
    )

data_actual_pose = np.load(npz_path)

actual_pos = data_actual_pose["base_eef_pos"]
actual_rotvec = data_actual_pose["base_eef_rot_axis_angle"]
episode_id_actual = data_actual_pose["episode_id"]
episode_timestep_actual = data_actual_pose["episode_timestep"]

available_actual_demo_ids = np.unique(episode_id_actual)
demo_id_actual = int(DEMO_ID_TO_PLOT) if "DEMO_ID_TO_PLOT" in globals() else int(available_actual_demo_ids[0])
if demo_id_actual not in set(int(x) for x in available_actual_demo_ids):
    raise ValueError(f"DEMO_ID_TO_PLOT={demo_id_actual} was not found. Available demos: {available_actual_demo_ids.tolist()}")
demo_mask_actual = episode_id_actual == demo_id_actual
demo_order_actual = np.argsort(episode_timestep_actual[demo_mask_actual])
demo_indices_actual = np.where(demo_mask_actual)[0][demo_order_actual]

path_xyz_actual = actual_pos[demo_indices_actual]
path_rotvec_actual = actual_rotvec[demo_indices_actual]

num_pose_markers = 25
marker_positions = np.unique(
    np.linspace(
        0,
        len(demo_indices_actual) - 1,
        min(num_pose_markers, len(demo_indices_actual)),
        dtype=int,
    )
)
marker_xyz = path_xyz_actual[marker_positions]
marker_rotvec = path_rotvec_actual[marker_positions]
marker_rot_mats = R.from_rotvec(marker_rotvec).as_matrix()

axis_scale = 0.025
x_axis_ends = marker_xyz + marker_rot_mats[:, :, 0] * axis_scale
y_axis_ends = marker_xyz + marker_rot_mats[:, :, 1] * axis_scale
z_axis_ends = marker_xyz + marker_rot_mats[:, :, 2] * axis_scale

actors = [
    vedo.Line(path_xyz_actual, c="black", lw=5, alpha=0.95),
    vedo.Points(marker_xyz, c="black", r=9, alpha=1.0),
    vedo.Arrows(marker_xyz, x_axis_ends, c="red", alpha=0.8),
    vedo.Arrows(marker_xyz, y_axis_ends, c="green", alpha=0.8),
    vedo.Arrows(marker_xyz, z_axis_ends, c="blue", alpha=0.8),
]

print("Actual pose-only 3D plot")
print("loaded:", npz_path)
print("demo id:", demo_id_actual)
print("demo samples:", len(demo_indices_actual))
print("orientation markers:", len(marker_xyz))
print("black line = actual end-effector path")
print("black dots = 25 evenly spaced poses")
print("red/green/blue arrows = local x/y/z orientation axes")

vedo.settings.default_backend = "vtk"

vedo.show(
    actors,
    axes=1,
    viewup="z",
    bg="white",
    title=f"Actual End-Effector Pose Only: Demo {demo_id_actual}",
)
